In [ ]:
# Clone Real-ESRGAN and enter the Real-ESRGAN
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
# Set up the environment
!pip install basicsr
!pip install facexlib
!pip install gfpgan

# -------新增的 start 如果版本有衝突那就拔掉這一大段----
!sed -i '/opencv-python/d' requirements.txt
!sed -i '/Pillow/d' requirements.txt
!sed -i '/torch>=1.7/d' requirements.txt
!sed -i '/torchvision/d' requirements.txt
!sed -i '/tqdm/d' requirements.txt
# -------新增的 end-------


!pip install -r requirements.txt
!python setup.py develop

# 下載漫畫文字segmentation模型的權重
!pip install ultralytics pillow opencv-python
!wget -O best.pt "https://huggingface.co/ShadowB/Manga109-panel-balloon-text-yolov26-segmentation/resolve/main/best.pt"





In [ ]:
import os
import zipfile
import shutil
import subprocess

import cv2
import numpy as np

from tqdm import tqdm
from ultralytics import YOLO



def yolo_inference(model, image_path):
    """回傳文字區域mask"""
    results = model.predict(
        source=image_path,
        imgsz=1920,
        conf=0.25,
        iou=0.7,
        retina_masks=True,
        verbose=False
    )

    for result in results:
        if result.masks is None:
            print(f"{image_path}:\n找不到文字\n")
            h, w = result.orig_shape
            text_mask = np.zeros((h, w), dtype=np.uint8)
            return text_mask


        # 建立黑底 mask
        h, w = result.orig_shape
        text_mask = np.zeros((h, w), dtype=np.uint8)

        for i, cls in enumerate(result.boxes.cls):
            class_id = int(cls)

            # 只保留 text
            if class_id == 1:
                mask = result.masks.data[i].cpu().numpy()

                # mask resize 到原圖大小
                mask = cv2.resize(
                    mask,
                    (w, h),
                    interpolation=cv2.INTER_NEAREST
                )

                # 疊加多個 text mask
                text_mask[mask > 0.5] = 255

    return text_mask



def blur(img, blur_amount=5):
    '''
    此方法來自 https://github.com/natethegreate/Screentone-Remover

    藉由模糊來去除網點
    '''

    if(blur_amount == 7):
        dst2 = cv2.GaussianBlur(img,(7,7),0)
        dst = cv2.bilateralFilter(dst2, 7, 80, 80)
    else:
        dst2 = cv2.GaussianBlur(img,(5,5),0)
        dst = cv2.bilateralFilter(dst2, 7, 10 * blur_amount, 80)

    return dst








ROOT = r'/content'

book_LUT = {} # {book_0 : 書名, book_1 : 書名}

valid_ext = [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"]

RAW_BOOK_DIR = os.path.join(ROOT, 'raw_books')
SOURCE_DIR = os.path.join(ROOT, 'sources')
RESULT_DIR = os.path.join(ROOT, 'results')
UPSCALE_DIR = os.path.join(ROOT, 'upscale')
SCREENTONE_REMOVE_DIR = os.path.join(ROOT, 'screentone_remove')

# 建立資料夾
if not os.path.exists(RAW_BOOK_DIR):
    os.makedirs(RAW_BOOK_DIR)

if not os.path.exists(SOURCE_DIR):
    os.makedirs(SOURCE_DIR)

if not os.path.exists(RESULT_DIR):
    os.makedirs(RESULT_DIR)

if not os.path.exists(RESULT_DIR):
    os.makedirs(UPSCALE_DIR)

if not os.path.exists(SCREENTONE_REMOVE_DIR):
    os.makedirs(SCREENTONE_REMOVE_DIR)

# 初始化清空
if os.path.exists(RAW_BOOK_DIR):
    shutil.rmtree(RAW_BOOK_DIR)

if os.path.exists(SOURCE_DIR):
    shutil.rmtree(SOURCE_DIR)

if os.path.exists(RESULT_DIR):
    shutil.rmtree(RESULT_DIR)

if os.path.exists(UPSCALE_DIR):
    shutil.rmtree(UPSCALE_DIR)

if os.path.exists(SCREENTONE_REMOVE_DIR):
    shutil.rmtree(SCREENTONE_REMOVE_DIR)

# 取得書名 (不取後面的附檔名)
book_names = [os.path.splitext(name)[0] for name in os.listdir(ROOT) if name.endswith('.zip')]


# 解壓縮
for i, book_name in enumerate(book_names):
    book_LUT[f"book_{i}"] = book_name

    src = os.path.join(ROOT, f'{book_name}.zip')
    trg = os.path.join(RAW_BOOK_DIR, book_name)

    # 解壓縮後會變成這樣 raw_books\他的書名\他的書名  因為我們是壓縮整個資料夾
    with zipfile.ZipFile(src, 'r') as zip_ref:
        zip_ref.extractall(trg)


# 建 SOURCE_DIR 裡的資料夾
for book_name in book_names:
    os.makedirs(os.path.join(SOURCE_DIR, book_name))


# 建 SCREENTONE_REMOVE_DIR 裡的資料夾
for book_name in book_names:
    os.makedirs(os.path.join(SCREENTONE_REMOVE_DIR, book_name))


# 把相片移動到 SOURCE_DIR 只複製相片防止資料夾裡有其他東西
for book_name in book_names:

    # raw_books\他的書名\他的書名
    book_dir = os.path.join(RAW_BOOK_DIR, book_name, book_name)

    # 處理每個資料夾 是相片就複製起來
    for file in os.listdir(book_dir):

        file_name, ext = os.path.splitext(file)

        # 是相片就複製起來
        if ext.lower() in valid_ext:
            src = os.path.join(book_dir,file)
            trg = os.path.join(SOURCE_DIR, book_name, file)
            shutil.copy(src, trg)



# 載入模型
text_model = YOLO("best.pt")


# 去除網點與取出文字區域
print("去除網點中...")
for book_name in tqdm(book_names):

    # 讀取相片位置
    image_dir = os.path.join(SOURCE_DIR, book_name)
    image_paths = [os.path.join(image_dir,file_name) for file_name in os.listdir(image_dir)]


    # 針對每一張圖片進行去除網點與取出文字區域
    for image_path in image_paths:
        image = cv2.imdecode(np.fromfile(file=image_path, dtype=np.uint8), cv2.IMREAD_COLOR)

        # 使用模糊來消除網點 要根據網點的大小與相片大小來決定 小顆的用3 中的用5 大的用7
        blur_image = blur(image, 5)

        # 使用yolo提取文字區域 (文字切得不好可以調dilate參數來調整區域範圍)
        text_mask = yolo_inference(text_model, image_path)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        text_mask = cv2.dilate(text_mask, kernel, iterations=0)

        # mask 二值化
        text_mask = text_mask > 0

        # 轉成跟原圖一樣變成三通道
        text_mask = np.stack([text_mask]*3, axis=-1)

        # 文字範圍就用原圖，不然就用 blur_image
        blend = np.where(text_mask, image, blur_image)


        # 存檔
        base_name = os.path.basename(image_path)
        _, ext = os.path.splitext(base_name)
        trg = os.path.join(SCREENTONE_REMOVE_DIR, book_name, base_name)
        cv2.imencode(ext, blend)[1].tofile(trg)



# 重新命名資料夾 因為REAL-ESRGAN 不能吃中文
for book_key, book_name in book_LUT.items():  # {book_0 : ??W, book_1 : ??W}
    old_name = os.path.join(SCREENTONE_REMOVE_DIR, book_name)
    new_name = os.path.join(SCREENTONE_REMOVE_DIR, book_key)
    os.rename(old_name, new_name)


# 改掉第三方模組的東西 新版的改方法名稱了
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.10/dist-packages/basicsr/data/degradations.py
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.11/dist-packages/basicsr/data/degradations.py
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.13/dist-packages/basicsr/data/degradations.py





# 推理每一本書 (要調整圖片放大倍率可以調-s那邊的數值[1,2,3,4])
print("推理中...")
for book_key, book_name in tqdm(book_LUT.items()):

    input_dir = os.path.join(SCREENTONE_REMOVE_DIR,book_key)
    output_dir = os.path.join(UPSCALE_DIR,book_key)
    subprocess.run([
        "python",
        "inference_realesrgan.py",
        "-i", input_dir,
        "-n", "realesr-general-x4v3",
        "-s", "1",
        "-o", output_dir,
        "--denoise_strength", "0.4",
        "--suffix", "out1x"
    ], check=True)


    # 壓縮資料夾
    src_dir = os.path.join(UPSCALE_DIR, book_key)
    out_put = os.path.join(RESULT_DIR, book_name)

    shutil.make_archive(
        out_put,   # 輸出檔名，不用加 .zip
        "zip",     # 輸出格式
        src_dir   # 要壓縮的資料夾
    )